In [2]:
import math

class Node:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0

        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Node(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        #обработка константы
        other = other if isinstance(other, Node) else Node(other)
        #прямой проход(результирующий узел)
        out = Node(self.data + other.data, (self, other), '+')
        #обратный проход(градиенты)
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        #привязываем градиент к результирующему узлу
        out._backward = _backward
        return out

    def __mul__(self, other):
        #обработка константы
        other = other if isinstance(other, Node) else Node(other)
        #прямой проход(результирующий узел)
        out = Node(self.data * other.data, (self, other), '*')
        #обратный проход(градиенты)
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        #привязываем градиент к результирующему узлу
        out._backward = _backward
        return out

    def relu(self):
        #прямой проход(результирующий узел)
        out = Node(self.data if self.data > 0 else 0.0, (self,), 'ReLU')#релу
        #обратный проход(градиенты)
        def _backward():
            self.grad += (self.data > 0) * out.grad #производная релу
        #привязываем градиент к результирующему узлу
        out._backward = _backward
        return out

    def sigmoid(self):
        #прямой проход(результирующий узел)
        s = 1 / (1 + math.exp(-self.data))#сигмоида
        out = Node(s, (self,), 'sigmoid')
        #обратный проход(градиенты)
        def _backward():
            self.grad += s * (1 - s) * out.grad #производная сигмоиды
        #привязываем градиент к результирующему узлу
        out._backward = _backward
        return out

    def backward(self):
        topo = [] #список узлов в топологическом порядке
        visited = set() 

        #DFS для построения топологической сортировки графа
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        #проход в обратном топологическом порядке для вычисления производной на каждом узле
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [3]:
import unittest
import math


class TestAutogradBasics(unittest.TestCase):

    def test_addition_grad(self):
        a = Node(2)
        b = Node(3)
        c = a + b
        c.backward()

        self.assertAlmostEqual(c.data, 5.0)
        self.assertAlmostEqual(a.grad, 1.0)
        self.assertAlmostEqual(b.grad, 1.0)

    def test_multiplication_grad(self):
        a = Node(2)
        b = Node(4)
        c = a * b
        c.backward()

        self.assertAlmostEqual(c.data, 8.0)
        self.assertAlmostEqual(a.grad, 4.0)
        self.assertAlmostEqual(b.grad, 2.0)

    def test_example(self):
        a = Node(2)
        b = Node(-3)
        c = Node(10)

        d = a + b * c
        e = d.relu()
        e.backward()

        self.assertEqual(a.data, 2)
        self.assertEqual(b.data, -3)
        self.assertEqual(c.data, 10)
        self.assertEqual(d.data, -28)
        self.assertEqual(e.data, 0)

        self.assertEqual(a.grad, 0)
        self.assertEqual(b.grad, 0)
        self.assertEqual(c.grad, 0)
        self.assertEqual(d.grad, 0)
        self.assertEqual(e.grad, 1)


class TestReLU(unittest.TestCase):

    def test_relu_positive(self):
        x = Node(5)
        y = x.relu()
        y.backward()

        self.assertAlmostEqual(y.data, 5.0)
        self.assertAlmostEqual(x.grad, 1.0)

    def test_relu_zero(self):
        x = Node(-1)
        y = x.relu()
        y.backward()

        self.assertAlmostEqual(y.data, 0.0)
        self.assertAlmostEqual(x.grad, 0.0)

    def test_relu_negative_vanishing_gradient(self):
        x = Node(-10)
        y = x.relu()
        y.backward()

        self.assertAlmostEqual(y.data, 0.0)
        self.assertAlmostEqual(x.grad, 0.0)


class TestSigmoid(unittest.TestCase):

    def test_sigmoid_at_zero(self):
        x = Node(0)
        y = x.sigmoid()
        y.backward()

        self.assertAlmostEqual(y.data, 0.5, places=6)
        self.assertAlmostEqual(x.grad, 0.25, places=6)

    def test_sigmoid_positive_saturation(self):
        x = Node(10)
        y = x.sigmoid()
        y.backward()

        self.assertAlmostEqual(y.data, 1.0, places=4)
        self.assertLess(x.grad, 1e-3)

    def test_sigmoid_negative_saturation(self):
        x = Node(-10)
        y = x.sigmoid()
        y.backward()

        self.assertAlmostEqual(y.data, 0.0, places=4)
        self.assertLess(x.grad, 1e-3)


class TestDeepGradients(unittest.TestCase):

    def test_deep_sigmoid_chain(self):
        x = Node(1.0)

        y = x
        for _ in range(10):
            y = y.sigmoid()

        y.backward()

        self.assertAlmostEqual(x.grad, 0.0, places=4)

    def test_deep_relu_chain(self):
        x = Node(1.0)

        y = x
        for _ in range(5):
            y = y.relu()

        y.backward()

        self.assertEqual(x.grad, 1.0)

        x = Node(-1.0)

        y = x
        for _ in range(5):
            y = y.relu()

        y.backward()

        self.assertEqual(x.grad, 0.0)


if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)


.c:\Users\Kostya\AppData\Local\Programs\Python\Python312\Lib\site-packages\jupyter_client\session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
..........
----------------------------------------------------------------------
Ran 11 tests in 0.006s

OK


Градиенты у сигмоиды и ReLu зануляются в разных случаях:

-ReLu обнуляет градиент в случае отрицательного или нулевого входа. В нашем примере из задания была как раз такая ситуация

-Сигмоида же имеет затухающие градиенты, эффект от которых накапливается в цепочке, если между активациями в линейном слое градиент сильно не растет(чем длинее цепь, тем меньше итоговый градиент). Также сигмоида может иметь очень близкую к нулю производную при очень большом или маленьком входе
